# Fase 2: Preprocesamiento e Ingeniería de Variables
## Sistema de Predicción de Factores de Riesgo en Adolescentes Salvadoreños (GSHS 2013)

**Propósito:** Este notebook toma el dataset limpio de la fase anterior y aplica técnicas de ingeniería de características. Se seleccionarán las variables predictoras (evitando la colinealidad y la fuga de datos), se imputarán los valores nulos, se codificarán las variables categóricas y se escalarán los datos para preparar las matrices finales de entrenamiento para los modelos de regresión y clasificación.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

print("Librerías de preprocesamiento cargadas exitosamente.")

Librerías de preprocesamiento cargadas exitosamente.


## 1. Carga de Datos y Separación de Variables (Q vs QN)

**¿Qué hace este bloque?**
Carga los datos procesados previamente. Luego, separa las variables predictoras ($X$) de las variables objetivo ($y$). Además, elimina las columnas redundantes y aquellas que causan *Data Leakage*.

**¿Por qué lo hacemos así?**
El dataset original contiene preguntas categóricas (`Q1` a `Q58`) y sus recodificaciones numéricas (`QN6` a `QN58`). Utilizar ambas simultáneamente generaría una **multicolinealidad perfecta**, arruinando los modelos (especialmente la regresión lineal o logística). Optaremos por conservar las preguntas originales (`Q`) y aplicarles codificación categórica más adelante, eliminando las `QN`.
Además, **es obligatorio** eliminar el Peso (`Q5`) y la Estatura (`Q4`) del conjunto de variables predictoras al predecir el IMC, así como la pregunta base del riesgo de salud mental (`Q25`).

In [2]:
# Carga del dataset limpio
df = pd.read_csv('../data/processed/SLV2013_Limpios_Targets.csv')

# Definición de las variables objetivo (Targets)
y_reg = df['IMC']
y_clf = df['Riesgo_Salud_Mental']

# Filtramos las columnas 'QN' y variables derivadas globales (suelen empezar con 'qn' minúscula)
columnas_a_eliminar = [col for col in df.columns if col.startswith('QN') or col.startswith('qn')]

# Agregamos las variables que causan Data Leakage y los Targets al filtro de eliminación
columnas_a_eliminar.extend(['Q4', 'Q5', 'Q25', 'IMC', 'Riesgo_Salud_Mental'])

# Variables auxiliares del diseño de la encuesta que no son predictoras
columnas_a_eliminar.extend(['weight', 'stratum', 'psu'])

# Matriz de características (Features)
X = df.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"Dimensiones de X (Predictoras): {X.shape}")
print(f"Dimensiones de y_reg (Target Regresión): {y_reg.shape}")
print(f"Dimensiones de y_clf (Target Clasificación): {y_clf.shape}")

Dimensiones de X (Predictoras): (1915, 46)
Dimensiones de y_reg (Target Regresión): (1915,)
Dimensiones de y_clf (Target Clasificación): (1915,)


## 2. Imputación de Valores Nulos

**¿Qué hace este bloque?**
Rellena los valores faltantes (`NaN`) en la matriz de características utilizando la estrategia de la moda (el valor más frecuente).

**¿Por qué lo hacemos así?**
Dado que la inmensa mayoría de nuestras variables predictoras (`Q1`, `Q2`, etc.) representan respuestas a encuestas de opción múltiple (categorías ordinales o nominales representadas por números enteros), imputar con la media generaría valores decimales inexistentes en las opciones de respuesta. La moda preserva la estructura de distribución categórica de la encuesta poblacional.

In [3]:
# Inicializamos el imputador con la estrategia 'most_frequent'
imputador = SimpleImputer(strategy='most_frequent')

# Aplicamos la imputación y reconstruimos el DataFrame
X_imputado = pd.DataFrame(imputador.fit_transform(X), columns=X.columns)

# Verificamos que no queden nulos
print(f"Total de valores nulos restantes en X: {X_imputado.isnull().sum().sum()}")

Total de valores nulos restantes en X: 0


## 3. Codificación de Variables Categóricas (One-Hot Encoding)

**¿Qué hace este bloque?**
Transforma las variables categóricas en múltiples columnas binarias (0 y 1).

**¿Por qué lo hacemos así?**
Aunque los datos de la encuesta son números (ej. 1, 2, 3, 4), no siempre tienen un orden matemático lógico (por ejemplo, 1=Manzana, 2=Naranja no implica que Naranja > Manzana). El `One-Hot Encoding` evita que los modelos asuman jerarquías falsas. Utilizamos el parámetro `drop_first=True` para evitar la trampa de las variables ficticias (colinealidad perfecta entre las nuevas columnas binarias generadas).

In [4]:
# Convertimos todas las columnas a tipo 'object' o 'category' temporalmente
# para forzar la creación de variables dummy
X_imputado = X_imputado.astype(str)

# Aplicamos One-Hot Encoding
X_codificado = pd.get_dummies(X_imputado, drop_first=True)

print(f"Dimensiones de X después de One-Hot Encoding: {X_codificado.shape}")

Dimensiones de X después de One-Hot Encoding: (1915, 220)


## 4. Escalado de Datos

**¿Qué hace este bloque?**
Estandariza el rango de las características utilizando `RobustScaler`. También imputamos los Targets en caso de que existan registros donde el IMC no pudo calcularse.

**¿Por qué lo hacemos así?**
Algoritmos como la Regresión Logística y la Regresión Lineal son sensibles a la magnitud de las variables. Aunque con One-Hot Encoding la mayoría de los datos son 0 y 1, aplicar un escalador asegura convergencia rápida durante el entrenamiento. Se prefiere `RobustScaler` porque utiliza la mediana y el rango intercuartílico, volviéndolo resistente a posibles valores atípicos (*outliers*) presentes en las encuestas.

In [5]:
# Escalado de características
escalador = RobustScaler()
X_escalado = pd.DataFrame(escalador.fit_transform(X_codificado), columns=X_codificado.columns)

# Imputación final en los Targets (Eliminamos filas donde el target sea nulo 
# para no entrenar con ruido, o imputamos la mediana para el IMC)
imputador_y = SimpleImputer(strategy='median')
y_reg = pd.Series(imputador_y.fit_transform(y_reg.values.reshape(-1, 1)).flatten(), name='IMC')

print("Escalado e imputación final de Targets completada.")

Escalado e imputación final de Targets completada.


## 5. Exportación de Matrices de Entrenamiento

**¿Qué hace este bloque?**
Guarda las matrices finales $X$ (escalada y codificada), $y\_reg$ y $y\_clf$ en la carpeta de procesados.

**¿Por qué lo hacemos así?**
Estas matrices ya están completamente listas para ser introducidas en los algoritmos de Machine Learning. Aislar este paso permite experimentar con distintos hiperparámetros en el siguiente notebook sin tener que recalcular el One-Hot Encoding y el escalado cada vez.

In [6]:
# Guardado de las matrices
X_escalado.to_csv('../data/processed/X_features.csv', index=False)
y_reg.to_csv('../data/processed/y_target_reg.csv', index=False)
y_clf.to_csv('../data/processed/y_target_clf.csv', index=False)

print("Matrices exportadas con éxito. Listas para la fase de Modelado.")

Matrices exportadas con éxito. Listas para la fase de Modelado.
